In [1]:
!git clone https://github.com/bowang-lab/MedSAM

Cloning into 'MedSAM'...
remote: Enumerating objects: 967, done.
remote: Total 967 (delta 0), reused 0 (delta 0), pack-reused 967 (from 1)
Receiving objects: 100% (967/967), 62.91 MiB | 14.27 MiB/s, done.
Resolving deltas: 100% (474/474), done.


In [2]:
%cd /content/MedSAM
!pip install -e .

/content/MedSAM
Obtaining file:///content/MedSAM
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.0/519.0 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 143.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 87.1 MB/s eta 0:00:00
  Running setup.py develop for medsam


In [18]:
from pathlib import Path

script_text = r'''
from __future__ import annotations

import argparse
import csv
import random
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
from skimage import transform
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from segment_anything import sam_model_registry


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def get_device(device_arg: str = "cuda:0"):
    if torch.cuda.is_available():
        return torch.device(device_arg)
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


@torch.no_grad()
def mean_dice(pred, target, num_classes, eps=1e-6, exclude_bg=True):
    vals = []
    start_c = 1 if exclude_bg else 0

    for b in range(pred.shape[0]):
        sample_dices = []
        for c in range(start_c, num_classes):
            pred_class = (pred[b] == c).float()
            targ_class = (target[b] == c).float()

            inter = (pred_class * targ_class).sum()
            denom = pred_class.sum() + targ_class.sum()

            if denom > 0:
                dice = (2 * inter + eps) / (denom + eps)
                sample_dices.append(dice)

        if len(sample_dices) > 0:
            vals.append(torch.stack(sample_dices).mean())

    if len(vals) == 0:
        return 0.0

    return torch.stack(vals).mean().item()


@torch.no_grad()
def dice_per_class(pred, target, num_classes, eps=1e-6):
    out = {}

    for c in range(1, num_classes):
        pred_class = (pred == c).float()
        targ_class = (target == c).float()

        inter = (pred_class * targ_class).sum(dim=(1, 2))
        denom = pred_class.sum(dim=(1, 2)) + targ_class.sum(dim=(1, 2))

        valid = denom > 0
        if valid.any():
            dice = (2 * inter[valid] + eps) / (denom[valid] + eps)
            out[c] = dice.mean().item()
        else:
            out[c] = float("nan")

    return out


def soft_dice_loss(logits, target, num_classes, eps=1e-6, exclude_bg=True):
    probs = torch.softmax(logits, dim=1)
    t1h = F.one_hot(target, num_classes).permute(0, 3, 1, 2).float()

    if exclude_bg:
        probs = probs[:, 1:]
        t1h = t1h[:, 1:]

    inter = (probs * t1h).sum(dim=(0, 2, 3))
    denom = probs.sum(dim=(0, 2, 3)) + t1h.sum(dim=(0, 2, 3))
    dice = (2 * inter + eps) / (denom + eps)
    return 1.0 - dice.mean()


def normalize_img(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)


def load_bbox_csv(csv_path: Path) -> Dict[str, np.ndarray]:
    mapping = {}
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            slice_file = row["slice_file"]
            box = np.array(
                [
                    float(row["bbox_x_min"]),
                    float(row["bbox_y_min"]),
                    float(row["bbox_x_max"]),
                    float(row["bbox_y_max"]),
                ],
                dtype=np.float32,
            )
            mapping[slice_file] = box
    return mapping


def build_items(img_dir: Path, mask_dir: Path, bbox_map: Dict[str, np.ndarray]) -> List[Tuple[Path, Path, np.ndarray]]:
    items = []
    for img_path in sorted(img_dir.glob("*__img.npy")):
        mask_path = mask_dir / img_path.name.replace("__img.npy", "__mask.npy")
        if not mask_path.exists():
            continue
        if img_path.name not in bbox_map:
            continue
        items.append((img_path, mask_path, bbox_map[img_path.name]))
    return items


def scale_box(box: np.ndarray, src_h: int, src_w: int, dst_h: int = 1024, dst_w: int = 1024) -> np.ndarray:
    x_min, y_min, x_max, y_max = box.astype(np.float32)
    sx = dst_w / float(src_w)
    sy = dst_h / float(src_h)
    return np.array([x_min * sx, y_min * sy, x_max * sx, y_max * sy], dtype=np.float32)


def jitter_bbox(box: np.ndarray, h: int, w: int, max_shift: int = 0) -> np.ndarray:
    if max_shift <= 0:
        return box.astype(np.float32)

    x_min, y_min, x_max, y_max = box.astype(np.int32)

    x_min = max(0, x_min - random.randint(0, max_shift))
    y_min = max(0, y_min - random.randint(0, max_shift))
    x_max = min(w - 1, x_max + random.randint(0, max_shift))
    y_max = min(h - 1, y_max + random.randint(0, max_shift))

    if x_max <= x_min:
        x_max = min(w - 1, x_min + 1)
    if y_max <= y_min:
        y_max = min(h - 1, y_min + 1)

    return np.array([x_min, y_min, x_max, y_max], dtype=np.float32)


class PromptedMultiClassDataset(Dataset):
    def __init__(
        self,
        img_dir: str,
        mask_dir: str,
        bbox_csv: str,
        num_classes: int = 5,
        bbox_shift: int = 0,
    ):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.num_classes = num_classes
        self.bbox_shift = bbox_shift

        bbox_map = load_bbox_csv(Path(bbox_csv))
        self.items = build_items(self.img_dir, self.mask_dir, bbox_map)

        if len(self.items) == 0:
            raise RuntimeError(f"No usable items found for {img_dir}, {mask_dir}, {bbox_csv}")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        img_path, mask_path, box_orig = self.items[idx]

        img = np.load(img_path, allow_pickle=True).astype(np.float32)
        gt = np.load(mask_path, allow_pickle=True).astype(np.int64)

        if img.ndim != 3:
            raise ValueError(f"Expected image (H,W,C), got {img.shape} for {img_path}")
        if img.shape[-1] < 3:
            raise ValueError(f"Need at least 3 channels, got {img.shape} for {img_path}")

        orig_h, orig_w = img.shape[:2]

        img = img[..., :3]
        img = normalize_img(img)

        img = transform.resize(
            img,
            (1024, 1024, 3),
            order=1,
            preserve_range=True,
            anti_aliasing=True,
        ).astype(np.float32)

        gt = transform.resize(
            gt,
            (1024, 1024),
            order=0,
            preserve_range=True,
            anti_aliasing=False,
        ).astype(np.int64)

        gt = np.clip(gt, 0, self.num_classes - 1)

        box = scale_box(box_orig, src_h=orig_h, src_w=orig_w, dst_h=1024, dst_w=1024)
        box = jitter_bbox(box, h=1024, w=1024, max_shift=self.bbox_shift)

        img_t = torch.tensor(img).permute(2, 0, 1).float()
        gt_t = torch.tensor(gt).long()
        box_t = torch.tensor(box).float()

        return {
            "image": img_t,
            "mask": gt_t,
            "box": box_t,
            "name": img_path.name,
        }


def compute_class_weights(mask_dir: Path, num_classes: int):
    counts = np.zeros(num_classes, dtype=np.float64)

    mask_paths = sorted(mask_dir.glob("*__mask.npy"))
    if len(mask_paths) == 0:
        raise RuntimeError(f"No masks found in {mask_dir}")

    for p in tqdm(mask_paths, desc=f"Counting labels in {mask_dir.name}", leave=False):
        m = np.load(p, allow_pickle=True).astype(np.int64)
        vals, cts = np.unique(m, return_counts=True)
        for v, c in zip(vals, cts):
            if 0 <= v < num_classes:
                counts[int(v)] += float(c)

    print("Raw class pixel counts:", {i: int(counts[i]) for i in range(num_classes)})

    counts = np.maximum(counts, 1.0)
    weights = counts.sum() / (num_classes * counts)
    weights[0] = min(weights[0], 1.0)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32), counts


@torch.no_grad()
def prediction_class_histogram(pred: torch.Tensor, num_classes: int):
    out = {}
    for c in range(num_classes):
        out[c] = int((pred == c).sum().item())
    return out


class PromptedMedSAMMultiClass(nn.Module):
    def __init__(self, image_encoder, prompt_encoder, num_classes: int = 5):
        super().__init__()
        self.image_encoder = image_encoder
        self.prompt_encoder = prompt_encoder
        self.num_classes = num_classes

        self.fuse = nn.Sequential(
            nn.Conv2d(256 + 256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        self.seg_head = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, image: torch.Tensor, box: torch.Tensor) -> torch.Tensor:
        image_embedding = self.image_encoder(image)

        if box.ndim == 2:
            box = box[:, None, :]

        sparse_embeddings, dense_embeddings = self.prompt_encoder(
            points=None,
            boxes=box,
            masks=None,
        )

        fused = torch.cat([image_embedding, dense_embeddings], dim=1)
        fused = self.fuse(fused)
        low_res_logits = self.seg_head(fused)

        logits = F.interpolate(
            low_res_logits,
            size=(image.shape[2], image.shape[3]),
            mode="bilinear",
            align_corners=False,
        )
        return logits


def run_epoch(
    model,
    loader,
    optimizer,
    device,
    num_classes,
    ce_criterion,
    w_ce=0.5,
    w_dice=0.5,
    scaler=None,
    train=True,
    use_amp=False,
):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_dice = 0.0
    count = 0

    per_class_sums = {c: 0.0 for c in range(1, num_classes)}
    per_class_counts = {c: 0 for c in range(1, num_classes)}

    pred_hist_total = {c: 0 for c in range(num_classes)}

    pbar = tqdm(loader, desc="train" if train else "val", leave=False)

    for batch in pbar:
        image = batch["image"].to(device, non_blocking=True)
        target = batch["mask"].to(device, non_blocking=True)
        box = batch["box"].to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            with torch.cuda.amp.autocast(enabled=use_amp):
                logits = model(image, box)
                ce = ce_criterion(logits, target)
                dl = soft_dice_loss(logits, target, num_classes=num_classes, exclude_bg=True)
                loss = w_ce * ce + w_dice * dl

            if train:
                optimizer.zero_grad(set_to_none=True)
                if scaler is not None and use_amp:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        pred = torch.argmax(logits.detach(), dim=1)
        bs = image.size(0)

        batch_dice = mean_dice(pred, target, num_classes=num_classes, exclude_bg=True)
        class_dice = dice_per_class(pred, target, num_classes=num_classes)

        total_loss += loss.item() * bs
        total_dice += batch_dice * bs
        count += bs

        hist = prediction_class_histogram(pred, num_classes)
        for c in range(num_classes):
            pred_hist_total[c] += hist[c]

        for c in range(1, num_classes):
            if not np.isnan(class_dice[c]):
                per_class_sums[c] += class_dice[c] * bs
                per_class_counts[c] += bs

    epoch_loss = total_loss / max(count, 1)
    epoch_dice = total_dice / max(count, 1)

    epoch_class_dice = {}
    for c in range(1, num_classes):
        if per_class_counts[c] > 0:
            epoch_class_dice[c] = per_class_sums[c] / max(per_class_counts[c], 1)
        else:
            epoch_class_dice[c] = float("nan")

    return epoch_loss, epoch_dice, epoch_class_dice, pred_hist_total


def save_checkpoint(path: Path, model, optimizer, epoch, args, best_val_dice):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "args": vars(args),
            "best_val_dice": best_val_dice,
            "num_classes": args.num_classes,
        },
        path,
    )


def main():
    print("RUNNING: train_medsam_multiclass_prompted.py")

    parser = argparse.ArgumentParser()

    parser.add_argument("--train_img_dir", type=str, required=True)
    parser.add_argument("--train_mask_dir", type=str, required=True)
    parser.add_argument("--train_bbox_csv", type=str, required=True)

    parser.add_argument("--val_img_dir", type=str, required=True)
    parser.add_argument("--val_mask_dir", type=str, required=True)
    parser.add_argument("--val_bbox_csv", type=str, required=True)

    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--work_dir", type=str, default="./work_dir_medsam_multiclass")
    parser.add_argument("--run_name", type=str, default="medsam_multiclass")

    parser.add_argument("--model_type", type=str, default="vit_b")
    parser.add_argument("--device", type=str, default="cuda:0")
    parser.add_argument("--num_epochs", type=int, default=15)
    parser.add_argument("--batch_size", type=int, default=2)
    parser.add_argument("--num_workers", type=int, default=0)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--weight_decay", type=float, default=1e-5)
    parser.add_argument("--num_classes", type=int, default=5)

    parser.add_argument("--bbox_shift", type=int, default=0)
    parser.add_argument("--freeze_prompt_encoder", action="store_true")
    parser.add_argument("--freeze_image_encoder", action="store_true")
    parser.add_argument("--use_amp", action="store_true")
    parser.add_argument("--seed", type=int, default=42)

    parser.add_argument("--w_ce", type=float, default=0.7)
    parser.add_argument("--w_dice", type=float, default=0.3)

    args = parser.parse_args()
    set_seed(args.seed)

    device = get_device(args.device)
    print("Using device:", device)

    train_ds = PromptedMultiClassDataset(
        img_dir=args.train_img_dir,
        mask_dir=args.train_mask_dir,
        bbox_csv=args.train_bbox_csv,
        num_classes=args.num_classes,
        bbox_shift=args.bbox_shift,
    )
    val_ds = PromptedMultiClassDataset(
        img_dir=args.val_img_dir,
        mask_dir=args.val_mask_dir,
        bbox_csv=args.val_bbox_csv,
        num_classes=args.num_classes,
        bbox_shift=0,
    )

    print("Train samples:", len(train_ds))
    print("Val samples:", len(val_ds))

    class_weights, raw_counts = compute_class_weights(Path(args.train_mask_dir), args.num_classes)
    print("Auto class weights for CE:", {i: round(float(class_weights[i]), 4) for i in range(args.num_classes)})

    class_weights = torch.tensor([0.02, 1.2, 0.5, 3.0, 1.5], dtype=torch.float32)
    print("Manual class weights for CE:", {i: float(class_weights[i]) for i in range(args.num_classes)})

    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    sam_model = sam_model_registry[args.model_type](checkpoint=args.checkpoint)
    model = PromptedMedSAMMultiClass(
        image_encoder=sam_model.image_encoder,
        prompt_encoder=sam_model.prompt_encoder,
        num_classes=args.num_classes,
    ).to(device)

    if args.freeze_prompt_encoder:
        for p in model.prompt_encoder.parameters():
            p.requires_grad = False

    if args.freeze_image_encoder:
        for p in model.image_encoder.parameters():
            p.requires_grad = False

    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=args.lr, weight_decay=args.weight_decay)
    scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)

    ce_criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

    run_dir = Path(args.work_dir) / args.run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    best_val_dice = -1.0

    for epoch in range(1, args.num_epochs + 1):
        print(f"\\nEpoch {epoch}/{args.num_epochs}")

        train_loss, train_dice, train_cls, train_hist = run_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            num_classes=args.num_classes,
            ce_criterion=ce_criterion,
            w_ce=args.w_ce,
            w_dice=args.w_dice,
            scaler=scaler,
            train=True,
            use_amp=args.use_amp,
        )

        val_loss, val_dice, val_cls, val_hist = run_epoch(
            model=model,
            loader=val_loader,
            optimizer=optimizer,
            device=device,
            num_classes=args.num_classes,
            ce_criterion=ce_criterion,
            w_ce=args.w_ce,
            w_dice=args.w_dice,
            scaler=None,
            train=False,
            use_amp=args.use_amp,
        )

        train_cls_str = " ".join([
            f"class{c}={train_cls[c]:.4f}" if not np.isnan(train_cls[c]) else f"class{c}=nan"
            for c in range(1, args.num_classes)
        ])
        val_cls_str = " ".join([
            f"class{c}={val_cls[c]:.4f}" if not np.isnan(val_cls[c]) else f"class{c}=nan"
            for c in range(1, args.num_classes)
        ])

        print(
            f"Epoch {epoch}: "
            f"train_loss={train_loss:.4f} "
            f"train_dice(excl_bg)={train_dice:.4f} "
            f"{train_cls_str}"
        )
        print(
            f"Epoch {epoch}: "
            f"val_loss={val_loss:.4f} "
            f"val_dice(excl_bg)={val_dice:.4f} "
            f"{val_cls_str}"
        )
        print("Train prediction histogram:", train_hist)
        print("Val prediction histogram:", val_hist)

        latest_ckpt = run_dir / "latest.pt"
        save_checkpoint(latest_ckpt, model, optimizer, epoch, args, best_val_dice)

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_ckpt = run_dir / "best_model.pt"
            save_checkpoint(best_ckpt, model, optimizer, epoch, args, best_val_dice)
            print(f"Saved new best checkpoint to {best_ckpt}")

    print("\\nTraining complete.")
    print("Best val dice:", best_val_dice)


if __name__ == "__main__":
    main()
'''

out_path = Path("/content/MedSAM/train_medsam_multiclass_prompted.py")
out_path.write_text(script_text)

print(f"Wrote file to: {out_path}")
print(f"File size: {out_path.stat().st_size} bytes")

Wrote script to: /content/MedSAM/train_medsam_multiclass_prompted_patched.py
File size: 20191 bytes


In [4]:
%cd /content/MedSAM

%mkdir val
%mkdir train
%cd val
%mkdir images
%mkdir masks
%cd /content/MedSAM/train
%mkdir images
%mkdir masks

/content/MedSAM
/content/MedSAM/val
/content/MedSAM/train


In [5]:
%cd /content/MedSAM

/content/MedSAM


In [38]:
#You will need to download the ckpt weights from the MedSAM repo
#You will need to upload train masks/images/bounding boxes
#You will need to upload val masks/images/bounding boxes

!python /content/MedSAM/train_medsam_multiclass_prompted.py \
  --train_img_dir /content/MedSAM/train/images \
  --train_mask_dir /content/MedSAM/train/masks \
  --train_bbox_csv /content/MedSAM/bboxes_train.csv \
  --val_img_dir /content/MedSAM/val/images \
  --val_mask_dir /content/MedSAM/val/masks \
  --val_bbox_csv /content/MedSAM/bboxes_val.csv \
  --checkpoint /content/MedSAM/work_dir/MedSAM/medsam_vit_b.pth \
  --work_dir /content/MedSAM/work_dir_medsam_multiclass \
  --run_name medsam_multiclass_patched_run1 \
  --model_type vit_b \
  --device cuda:0 \
  --num_epochs 15 \
  --batch_size 4 \
  --num_workers 0 \
  --lr 1e-4 \
  --weight_decay 1e-5 \
  --num_classes 5 \
  --bbox_shift 0 \
  --freeze_image_encoder \
  --use_amp

RUNNING: train_medsam_multiclass_prompted_patched.py
Using device: cuda:0
Train samples: 356
Val samples: 376
Raw class pixel counts: {0: 21542005, 1: 87991, 2: 422292, 3: 36866, 4: 86846}
Auto class weights for CE: {0: 0.0044, 1: 1.084, 2: 0.2259, 3: 2.5873, 4: 1.0983}
Manual class weights for CE: {0: 0.019999999552965164, 1: 1.2000000476837158, 2: 0.5, 3: 3.0, 4: 1.5}
/content/MedSAM/train_medsam_multiclass_prompted_patched.py:552: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=args.use_amp)
\nEpoch 1/15
train:   0% 0/89 [00:00<?, ?it/s]/content/MedSAM/train_medsam_multiclass_prompted_patched.py:390: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Epoch 1: train_loss=1.0964 train_dice(excl_bg)=0.1047 class1=0.0951 class2=0.2503 class3=0.03

In [46]:
from pathlib import Path
import csv

import numpy as np
from skimage import transform
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from segment_anything import sam_model_registry


#You will need to upload the bounding boxes that are inferred for the unlabeled train
BBOX_CSV = Path("/content/MedSAM/bboxes.csv")
UNLABELED_ROOT = Path("/content/MedSAM/unlabeled")
OUT_DIR = Path("/content/MedSAM/unlabeled_masks")

BEST_CKPT = Path("/content/MedSAM/work_dir_medsam_multiclass/medsam_multiclass_patched_run1/best_model.pt")
BASE_CKPT = Path("/content/MedSAM/work_dir/MedSAM/medsam_vit_b.pth")

NUM_CLASSES = 5
MODEL_TYPE = "vit_b"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))

# optional: save smaller files
SAVE_FLOAT16 = True


def normalize_img(img: np.ndarray) -> np.ndarray:
    img = img.astype(np.float32)
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)


def load_bbox_rows(csv_path: Path):
    rows = []
    with open(csv_path, "r", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append(row)
    return rows


def scale_box(box: np.ndarray, src_h: int, src_w: int, dst_h: int = 1024, dst_w: int = 1024):
    x_min, y_min, x_max, y_max = box.astype(np.float32)
    sx = dst_w / float(src_w)
    sy = dst_h / float(src_h)
    return np.array([x_min * sx, y_min * sy, x_max * sx, y_max * sy], dtype=np.float32)


def resolve_image_path(unlabeled_root: Path, slice_file: str) -> Path:
    candidates = [
        unlabeled_root / slice_file,
        unlabeled_root / "images" / slice_file,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find image for {slice_file} in {unlabeled_root}")


class PromptedMedSAMMultiClass(nn.Module):
    def __init__(self, image_encoder, prompt_encoder, num_classes: int = 5):
        super().__init__()
        self.image_encoder = image_encoder
        self.prompt_encoder = prompt_encoder
        self.num_classes = num_classes

        self.fuse = nn.Sequential(
            nn.Conv2d(256 + 256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
        )

        self.seg_head = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, image: torch.Tensor, box: torch.Tensor) -> torch.Tensor:
        image_embedding = self.image_encoder(image)

        if box.ndim == 2:
            box = box[:, None, :]

        sparse_embeddings, dense_embeddings = self.prompt_encoder(
            points=None,
            boxes=box,
            masks=None,
        )

        fused = torch.cat([image_embedding, dense_embeddings], dim=1)
        fused = self.fuse(fused)
        low_res_logits = self.seg_head(fused)

        logits = F.interpolate(
            low_res_logits,
            size=(image.shape[2], image.shape[3]),
            mode="bilinear",
            align_corners=False,
        )
        return logits


assert BBOX_CSV.exists(), f"Missing bbox csv: {BBOX_CSV}"
assert UNLABELED_ROOT.exists(), f"Missing unlabeled root: {UNLABELED_ROOT}"
assert BEST_CKPT.exists(), f"Missing trained checkpoint: {BEST_CKPT}"
assert BASE_CKPT.exists(), f"Missing base checkpoint: {BASE_CKPT}"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Using device:", DEVICE)

sam_model = sam_model_registry[MODEL_TYPE](checkpoint=str(BASE_CKPT))
model = PromptedMedSAMMultiClass(
    image_encoder=sam_model.image_encoder,
    prompt_encoder=sam_model.prompt_encoder,
    num_classes=NUM_CLASSES,
).to(DEVICE)

ckpt = torch.load(BEST_CKPT, map_location="cpu")
model.load_state_dict(ckpt["model"])
model.eval()

rows = load_bbox_rows(BBOX_CSV)
assert len(rows) > 0, "No bbox rows found in bboxes.csv"

saved = 0

for row in tqdm(rows, desc="SAM soft inference on unlabeled"):
    slice_file = row["slice_file"]
    image_path = resolve_image_path(UNLABELED_ROOT, slice_file)

    img_orig = np.load(image_path, allow_pickle=True).astype(np.float32)

    if img_orig.ndim != 3:
        raise ValueError(f"Expected image (H,W,C), got {img_orig.shape} for {image_path}")
    if img_orig.shape[-1] < 3:
        raise ValueError(f"Need at least 3 channels, got {img_orig.shape} for {image_path}")

    orig_h, orig_w = img_orig.shape[:2]

    bbox = np.array(
        [
            float(row["bbox_x_min"]),
            float(row["bbox_y_min"]),
            float(row["bbox_x_max"]),
            float(row["bbox_y_max"]),
        ],
        dtype=np.float32,
    )
    bbox_1024 = scale_box(bbox, src_h=orig_h, src_w=orig_w)

    img = img_orig[..., :3]
    img = normalize_img(img)
    img_1024 = transform.resize(
        img,
        (1024, 1024, 3),
        order=1,
        preserve_range=True,
        anti_aliasing=True,
    ).astype(np.float32)

    x = torch.tensor(img_1024).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
    box = torch.tensor(bbox_1024).unsqueeze(0).float().to(DEVICE)

    with torch.no_grad():
        logits = model(x, box)
        probs_1024 = torch.softmax(logits, dim=1)[0].cpu().numpy().astype(np.float32)  # (C,1024,1024)

    probs_orig = np.zeros((NUM_CLASSES, orig_h, orig_w), dtype=np.float32)
    for c in range(NUM_CLASSES):
        probs_orig[c] = transform.resize(
            probs_1024[c],
            (orig_h, orig_w),
            order=1,
            preserve_range=True,
            anti_aliasing=True,
        ).astype(np.float32)

    # renormalize across channels after resize
    probs_orig = np.clip(probs_orig, 1e-8, None)
    probs_orig = probs_orig / probs_orig.sum(axis=0, keepdims=True)

    if SAVE_FLOAT16:
        probs_orig = probs_orig.astype(np.float16)

    out_path = OUT_DIR / slice_file.replace("__img.npy", "__soft.npy")
    np.save(out_path, probs_orig)
    saved += 1

print(f"Done. Saved {saved} soft masks to {OUT_DIR}")

Using device: cuda:0


SAM soft inference on unlabeled: 100%|██████████| 1391/1391 [10:17<00:00,  2.25it/s]

Done. Saved 1391 soft masks to /content/MedSAM/unlabeled_masks


In [ ]:
from pathlib import Path
import shutil


folder_to_zip = Path("/content/MedSAM/unlabeled_masks")
zip_base = Path("/content/MedSAM/unlabeled_masks")


zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(folder_to_zip))

print("Created:", zip_path)


from google.colab import files
files.download(zip_path)